In [ ]:
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate
!pip install -q evaluate sacrebleu rouge-score trulens openai
!pip install -q sentencepiece protobuf scikit-learn pandas numpy

In [ ]:
import os
import gc
import json
import torch
import random
import numpy as np
import pandas as pd

from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)

from trl import SFTTrainer

import evaluate

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
)


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IS_KAGGLE = os.path.exists("/kaggle/working") or os.path.exists("/kaggle/input")
WORKDIR = "/kaggle/working" if IS_KAGGLE else os.getcwd()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Kaggle GPUs are commonly T4/P100, so keep the default precision conservative.
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
BNB_COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

MODELS = {
    "SmolLM2-360M": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "Qwen3-0.6B": "Qwen/Qwen3-0.6B",
    "Gemma-3-270m": "google/gemma-3-270m-it",
}

MAX_LENGTH = 512

OUTPUT_DIR = os.path.join(WORKDIR, "security_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Kaggle runtime detected: {IS_KAGGLE}")
print(f"Using output directory: {OUTPUT_DIR}")
print(f"Attention precision: {'bf16' if USE_BF16 else 'fp16'}")

In [ ]:
TASK_TYPE_MAP = {
    "trendyol": "generation",
    "security_qna": "generation",
    "purple_team": "generation",
    "soc_audit": "generation",
    "syslog": "generation",
    "multilingual_jailbreak": "classification",
    "cve_llm": "generation",
    "mitre_stix": "generation",
    "attackqa": "generation",
}

In [ ]:
print("Loading Security Domain Datasets...")

datasets_dict = {}

In [ ]:
datasets_dict["trendyol"] = load_dataset(
    "Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset"
)

datasets_dict["security_qna"] = load_dataset(
    "Mr-Vicky-01/Security-QnA"
)

datasets_dict["soc_audit"] = load_dataset(
    "harleygilpin/soc-audit-11k"
)

datasets_dict["syslog"] = load_dataset(
    "witfoo/syslog-to-artifact"
)

datasets_dict["multilingual_jailbreak"] = load_dataset(
    "darkknight25/Multilingual-Jailbreak-Dataset"
)

datasets_dict["cve_llm"] = load_dataset(
    "morpheuslord/cve-llm-training"
)

datasets_dict["mitre_stix"] = load_dataset(
    "jason-oneal/mitre-stix-cve-exploitdb-dataset"
)


In [ ]:
datasets_dict["purple_team"] = load_dataset(
    "Canstralian/Purple-Team-Cybersecurity-Dataset"
)

datasets_dict["attackqa"] = load_dataset(
    "sambanovasystems/attackqa"
)

In [ ]:
# ============================================================
# DATA CLEANING
# ============================================================

def clean_text(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    text = " ".join(text.split())

    return text.strip()

In [ ]:
def build_prompt(instruction, context, input_text, output_text=None):

    prompt = f"""
<instruction>
{instruction}
</instruction>

<context>
{context}
</context>

<input>
{input_text}
</input>

<output>
"""

    if output_text is not None:
        prompt += f"{output_text}"

    return prompt.strip()


In [ ]:
def safe_get(example, keys):

    for k in keys:
        if k in example and example[k] is not None:
            return str(example[k])

    return ""


In [ ]:
def format_generation(example):

    instruction = safe_get(
        example,
        ["instruction", "question", "prompt"]
    )

    context = safe_get(
        example,
        ["context", "passage", "description"]
    )

    input_text = safe_get(
        example,
        ["input", "query", "text"]
    )

    output = safe_get(
        example,
        ["output", "answer", "response", "label"]
    )

    return {
        "text": build_prompt(
            instruction,
            context,
            input_text,
            output
        )
    }

In [ ]:
def format_classification(example):

    text = safe_get(
        example,
        ["prompt", "text", "query"]
    )

    label = safe_get(
        example,
        ["label", "answer"]
    )

    instruction = """
Classify the following cybersecurity prompt as:
SAFE or UNSAFE
"""

    return {
        "text": build_prompt(
            instruction,
            "",
            text,
            label
        )
    }


In [ ]:
processed_train_datasets = []
eval_datasets = {}

for name, ds in datasets_dict.items():

    split = "train" if "train" in ds else list(ds.keys())[0]

    data = ds[split]

    data = data.shuffle(seed=SEED)

    small_data = data.select(
        range(min(1000, len(data)))
    )

    task_type = TASK_TYPE_MAP[name]

    if task_type == "generation":
        formatted = small_data.map(format_generation)

    else:
        formatted = small_data.map(format_classification)

    eval_datasets[name] = formatted

    # ========================================================
    # EXCLUDE UNSEEN DATASETS FROM TRAINING
    # ========================================================

    if name not in ["purple_team", "attackqa"]:

        processed_train_datasets.append(
            formatted
        )


In [ ]:
merged_train_dataset = concatenate_datasets(
    processed_train_datasets
)

print("Merged Train Size:", len(merged_train_dataset))


In [ ]:
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

In [ ]:
def compute_generation_metrics(preds, refs):

    bleu_score = bleu.compute(
        predictions=preds,
        references=[[r] for r in refs]
    )["score"]

    rouge_scores = rouge.compute(
        predictions=preds,
        references=refs
    )

    pred_tokens = " ".join(preds).split()
    ref_tokens = " ".join(refs).split()

    overlap = len(
        set(pred_tokens) & set(ref_tokens)
    )

    precision = overlap / (len(pred_tokens) + 1e-8)
    recall = overlap / (len(ref_tokens) + 1e-8)

    f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return {
        "BLEU": bleu_score,
        "ROUGE-1": rouge_scores["rouge1"],
        "ROUGE-2": rouge_scores["rouge2"],
        "ROUGE-L": rouge_scores["rougeL"],
        "F1": f1,
    }


In [ ]:
def compute_classification_metrics(preds, refs):

    return {
        "Accuracy": accuracy_score(refs, preds),
        "Macro-F1": f1_score(
            refs,
            preds,
            average="macro"
        ),
        "Precision": precision_score(
            refs,
            preds,
            average="macro"
        ),
        "Recall": recall_score(
            refs,
            preds,
            average="macro"
        ),
    }

In [ ]:
def load_model(model_name):

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=BNB_COMPUTE_DTYPE,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "v_proj",
        ],
    )

    model = get_peft_model(
        model,
        lora_config
    )

    return model, tokenizer

In [ ]:
def generate(model, tokenizer, prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=0.0,
        )

    decoded = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return decoded

In [ ]:
def evaluate_dataset(
    model,
    tokenizer,
    dataset_name,
    dataset,
):

    preds = []
    refs = []

    task_type = TASK_TYPE_MAP[dataset_name]

    subset = dataset.select(
        range(min(50, len(dataset)))
    )

    for ex in subset:

        text = ex["text"]

        split_prompt = text.split("<output>")[0]

        ref = text.split("<output>")[-1].strip()

        pred = generate(
            model,
            tokenizer,
            split_prompt
        )

        pred = pred.split("<output>")[-1].strip()

        preds.append(pred)
        refs.append(ref)

    if task_type == "generation":

        metrics = compute_generation_metrics(
            preds,
            refs
        )

    else:

        metrics = compute_classification_metrics(
            preds,
            refs
        )

    return metrics


In [ ]:
all_results = {}

for model_key, model_path in MODELS.items():

    print("=" * 80)
    print("MODEL:", model_key)
    print("=" * 80)

    model, tokenizer = load_model(model_path)

    # ========================================================
    # BEFORE FT
    # ========================================================

    before_results = {}

    for ds_name, ds in eval_datasets.items():

        print("Before FT:", ds_name)

        metrics = evaluate_dataset(
            model,
            tokenizer,
            ds_name,
            ds
        )

        before_results[ds_name] = metrics

    # ========================================================
    # TRAINING
    # ========================================================

    training_args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, f"{model_key}_security"),
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=20,
        save_strategy="no",
        bf16=USE_BF16,
        fp16=not USE_BF16,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=merged_train_dataset,
        dataset_text_field="text",
        args=training_args,
        max_seq_length=MAX_LENGTH,
    )

    trainer.train()

    # ========================================================
    # AFTER FT
    # ========================================================

    after_results = {}

    for ds_name, ds in eval_datasets.items():

        print("After FT:", ds_name)

        metrics = evaluate_dataset(
            model,
            tokenizer,
            ds_name,
            ds
        )

        after_results[ds_name] = metrics

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    all_results[model_key] = {
        "before_ft": before_results,
        "after_ft": after_results,
    }

    with open(
        f"{OUTPUT_DIR}/{model_key}_results.json",
        "w"
    ) as f:

        json.dump(
            all_results[model_key],
            f,
            indent=4
        )

    del model
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
rows = []

for model_name, results in all_results.items():

    for stage in ["before_ft", "after_ft"]:

        for dataset_name, metrics in results[stage].items():

            row = {
                "Model": model_name,
                "Stage": stage,
                "Dataset": dataset_name,
            }

            row.update(metrics)

            rows.append(row)

df = pd.DataFrame(rows)

df.to_csv(
    f"{OUTPUT_DIR}/security_results.csv",
    index=False
)

print(df.head())